# Step 2.3: the strategy specification and the signal frame

This notebook is the check of step 2.3. A signal strategy is Polars expressions for its features and signals, and an execution template for what it does with them. `to_spec()` writes the template as a `StrategySpec` document: the one the strategy builder, the SDK, and the engine share, which the JSON schema generated from the engine's Rust types validates (`python/fefat/src/fefat/strategy/strategy_spec.schema.json`).

The notebook builds the example strategy of `docs/masterplan/02-engine-and-data.md` §5 and prints its specification, evaluates its signal frame on your bars, and runs the strategy of the first backtest (`notebooks/02-first-backtest.ipynb`) as a template through the reference engine, whose trades must equal that notebook's. Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and changes nothing in it. Above each cell is what it should print on this PC.

## The product and the month

The same choice as `02-first-backtest.ipynb`: MES when your store holds its 1-minute bars, and the last 23 sessions of the contract that was the front for the most sessions in the last 120 days.

Expected on this PC: `MES from databento: MESU2026, 23 sessions from 2026-08-12 to 2026-09-14`.

In [ ]:
import os
import time
from datetime import time as clock
from datetime import timedelta

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.saved_instruments import instrument_table
from fefat.market.schema import Timeframe
from fefat.reference.engine import ReferenceEngine
from fefat.reference.hours import CalendarHours
from fefat.reference.ledger import Trade, trades_of
from fefat.reference.settings import RunSettings
from fefat.reference.strategy import decisions_of
from fefat.reference.tape import tape_of_bars
from fefat.reference.template import TemplateStrategy
from fefat.reference.timeline import RunResult
from fefat.reference.units import instrument_parameters, money_text

paths = resolve_paths(os.environ)
instruments = instrument_table(paths.data_dir)
listed = ff.market.catalog().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
daily = ff.market.load(
    ROOT, "1d", LAST - timedelta(days=120), LAST, fidelity="1m", provider=PROVIDER
).collect()
fronts = (
    daily.group_by("contract")
    .agg(sessions=pl.len(), last=pl.col("trade_date").max())
    .sort("sessions", "last", descending=True)
)
CONTRACT = fronts.item(0, "contract")
held = daily.filter(pl.col("contract") == CONTRACT).get_column("trade_date").sort().tail(23)
START, END = held.min(), held.max()
five_minutes = ff.market.load(CONTRACT, "5m", START, END, fidelity="1m", provider=PROVIDER)
minutes = ff.market.load(CONTRACT, "1m", START, END, fidelity="1m", provider=PROVIDER).collect()
print(f"{ROOT} from {PROVIDER}: {CONTRACT}, {counted(held.len(), 'session')} from {START} to {END}")

## The example strategy

The strategy of 02 §5, on the product of your store where the example names ES, with inline Polars expressions where the RSI and the ATR of `fefat.indicators` will be (Phase 3). The RSI is 0/0, NaN, on a stretch without a change, and the signal frame refuses NaN, which Polars compares as larger than any number, so the expression makes it null with `fill_nan(None)`. Two spellings differ from 02 §5: a target's share of the position is `fraction=`, since names spell words out, and `ff.Stop(atr=1.5)` reads the feature named `atr`.

Expected: the specification as its model prints it, then as its JSON document: an `atr` stop of 1.5, targets at 1 R, closing half the position, and at 2 R, breakeven after the first target, entries from 08:30 to 14:30 Chicago, flat by 15:05, three trades a day at most, and one lot, with the rules it does not use left out.

In [ ]:
change = pl.col("close").diff()
gain, loss = change.clip(lower_bound=0), (-change).clip(lower_bound=0)
rsi = 100 - 100 / (1 + gain.ewm_mean(alpha=1 / 14) / loss.ewm_mean(alpha=1 / 14))
before = pl.col("close").shift(1)
true_range = pl.max_horizontal(
    pl.col("high") - pl.col("low"), (pl.col("high") - before).abs(), (pl.col("low") - before).abs()
)
strat = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={"rsi": rsi.fill_nan(None), "atr": true_range.ewm_mean(alpha=1 / 14)},
    long_entry=pl.col("rsi") < 30,
    short_entry=pl.col("rsi") > 70,
    execution=ff.Execution(
        entry=ff.Market(),
        stop=ff.Stop(atr=1.5),
        targets=[ff.Target(r=1, fraction=0.5), ff.Target(r=2)],
        breakeven=ff.Breakeven(after_target=1),
        time=ff.TimeRules(windows=[("08:30", "14:30")], flat_by="15:05", tz="America/Chicago"),
        limits=ff.Limits(max_trades_per_day=3),
        sizing=ff.Fixed(1),
    ),
)
print(strat.to_spec())
print(strat.to_spec().document_json())

## Its signal frame

The example's signal frame on the month's 5-minute bars: one row per bar close, with the bars' `ts_close`, `trade_date`, `contract`, and `adj_offset`, the features, and the four signals; the RSI is null while it warms up, and a null signal is no signal. From step 2.6 the engine reads this frame beside the specification. The reference engine runs only the basic subset of a template, below; this strategy's ATR stop, two targets, breakeven, and entry window arrive with step 2.8.

Expected on this PC: 6,576 rows, the counts of long and short entry signals, and the frame's first rows.

In [ ]:
frame = strat.signals(five_minutes).collect()
longs, shorts = frame.get_column("long_entry").sum(), frame.get_column("short_entry").sum()
print(f"Signal frame: {counted(frame.height, 'row')}; entry signals: {longs} long, {shorts} short")
display(frame.head())

## The first backtest's strategy as a template

The strategy of `02-first-backtest.ipynb` as a `SignalStrategy`: buy one lot when a 5-minute close crosses above the mean of the last 20 closes between 08:30 and 15:00 Chicago, the entry window written into `long_entry` until windows arrive in step 2.8; a stop 12 ticks under the fill and a target 24 ticks over it; and that notebook's exit, which closes a long at the first close at or after 15:05 Chicago, as the signal exit `long_exit`. The reference engine runs it through `TemplateStrategy` on the same 1-minute bars, so its trades must be that notebook's.

Expected on this PC: 224 fills and 112 trades, with the total before fees that `02-first-backtest.ipynb` prints, and no position at the end.

In [ ]:
instrument = instruments.get(ROOT)
zone = instrument.session_calendar().hours.time_zone
sessions = instruments.sessions_of(ROOT, paths.cache_dir)
parameters = instrument_parameters(instrument)
one_minute = Timeframe.parse("1m")
tape = tape_of_bars(minutes)
hours = CalendarHours.around(sessions, tape[0].starts_at(), tape[-1].available_at())
local_close = (
    pl.from_epoch("ts_close", time_unit="ns")
    .dt.replace_time_zone("UTC")
    .dt.convert_time_zone(zone)
    .dt.time()
)
cross_up = (pl.col("close") > pl.col("mean")) & (
    pl.col("close").shift(1) <= pl.col("mean").shift(1)
)
in_hours = pl.col("local_close").is_between(clock(8, 30), clock(15, 0), closed="left")
bracket_and_size = {
    "entry": ff.Market(),
    "stop": ff.Stop(ticks=12),
    "targets": [ff.Target(ticks=24)],
    "sizing": ff.Fixed(1),
}


def run(strategy: ff.SignalStrategy) -> RunResult:
    """The strategy through the reference engine's template on the month's 1-minute bars."""
    signals = strategy.signals(five_minutes).collect()
    template = TemplateStrategy(strategy.to_spec(), signals, parameters, sessions)
    engine = ReferenceEngine(RunSettings.for_fidelity(one_minute), parameters, one_minute, hours)
    return engine.run(tape, decisions_of(signals), template.on_bar)


first_backtest = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={"mean": pl.col("close").rolling_mean(20), "local_close": local_close},
    long_entry=cross_up & in_hours,
    short_entry=pl.lit(False),
    long_exit=pl.col("local_close") >= clock(15, 5),
    execution=ff.Execution(**bracket_and_size),
)
started = time.perf_counter()
result = run(first_backtest)
seconds = time.perf_counter() - started
TRADES = trades_of(result.fills)
total = money_text(sum(trade.pnl for trade in TRADES))
print(
    f"Fills: {len(result.fills)}; trades: {len(TRADES)}; total before fees: {total} {instrument.currency}"
)
print(f"Run: {seconds:.1f} s for {len(tape):,} bars; position at the end: {result.position} lots")

## The same strategy with a flat-by time

The template's own flat-by time in place of the signal exit: `ff.TimeRules(flat_by="15:05", tz=zone)` closes the position at the first close at or after 15:05 on each trade date's own day, and places no entry after it that trade date. The two read a holiday differently. A halt day has no trade date of its own, since its trading belongs to the next trade date's session, so with flat-by a position held into a holiday's halt stays open until that trade date's flat-by time unless its bracket closes it first, where the signal exit closes it at the first close after the halt. Step 2.8 decides which reading flat-by keeps (`docs/masterplan/BACKLOG.md`).

Expected on this PC: 112 trades, of which one differs from the signal exit's, on Labor Day: the long opened on the morning of 2026-09-07, which the signal exit closes at the first close after the halt, and which flat-by leaves to its stop that evening.

In [ ]:
def local_time(instant: int) -> str:
    """An instant as the date and time of the instrument's zone."""
    utc = pl.from_epoch(pl.lit(instant), time_unit="ns").dt.replace_time_zone("UTC")
    return pl.select(utc.dt.convert_time_zone(zone).dt.strftime("%Y-%m-%d %H:%M")).item()


with_flat_by = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={"mean": pl.col("close").rolling_mean(20), "local_close": local_close},
    long_entry=cross_up & in_hours,
    short_entry=pl.lit(False),
    execution=ff.Execution(**bracket_and_size, time=ff.TimeRules(flat_by="15:05", tz=zone)),
)
FLAT_BY_TRADES = trades_of(run(with_flat_by).fills)
total = money_text(sum(trade.pnl for trade in FLAT_BY_TRADES))
print(
    f"Trades with flat-by: {len(FLAT_BY_TRADES)}; total before fees: {total} {instrument.currency}"
)


def without_reason(trade: Trade) -> dict[str, object]:
    """A trade as both runs can make it: the signal exit and flat-by close a trade at the same
    close under exit reasons of their own (`signal_exit`, `flat_by`)."""
    return trade.model_dump(exclude={"exit_reason"})


runs = (("the signal exit", TRADES, FLAT_BY_TRADES), ("flat-by", FLAT_BY_TRADES, TRADES))
for name, trades, others in runs:
    shared = [without_reason(trade) for trade in others]
    for trade in (trade for trade in trades if without_reason(trade) not in shared):
        entry, exit = local_time(trade.entry_ts), local_time(trade.exit_ts)
        print(f"  only with {name}: entry {entry}, exit {exit} at its {trade.exit_reason}")

## Your checks

- The specification printed under *The example strategy* reads like the builder form you expect: one object with its `kind` for each rule, and the rules the strategy does not use left out.
- The trades and the total of the first backtest's strategy as a template are those `02-first-backtest.ipynb` prints: the template runs the strategy that notebook wrote by hand. The test of this notebook compares the two trade lists trade by trade on the synthetic bars.
- With flat-by, a trade differs only where a halt comes before the flat-by time, on Labor Day this month; say at step 2.8 which reading you want flat-by to keep.